In [ ]:
!pip install pulp

Modelo Inteiro - Gestão Florestal

In [ ]:
import pulp
import random

# ==========================================
# 2. DEFINIÇÃO DOS PARÂMETROS (Dados de Teste)
# ==========================================
N = 3
M = {1: 4, 2: 3, 3: 5}

random.seed(42)

P = {}   # Lucro
W = {}   # Mão-de-obra
T = {}   # Transporte
F = {}   # Madeira fornecida
A = {}   # Área ocupada
S = {}   # Estacas e mourões

for i in range(1, N + 1):
    for j in range(1, M[i] + 1):
        P[(i, j)] = random.randint(50, 150)
        W[(i, j)] = random.randint(5, 20)
        T[(i, j)] = random.randint(2, 10)
        F[(i, j)] = random.randint(30, 100)
        A[(i, j)] = random.randint(10, 50)
        S[(i, j)] = random.randint(15, 60)

# Limites Globais e por Floresta
MO_max = 120
NC_max = 60
SL_min = 250
EM_min = 180
A_max = {1: 100, 2: 80, 3: 120}

# ==========================================
# EXTRA: EXIBIÇÃO DA TABELA DE PARÂMETROS
# ==========================================
print("=" * 72)
print(f"{'TABELA DE PARÂMETROS DOS LOTES':^72}")
print("=" * 72)
# Cabeçalho da tabela
print(f"{'Floresta':<10} | {'Lote':<6} | {'Lucro (P)':<9} | {'Mão-Obra (W)':<12} | {'Transp. (T)':<11} | {'Madeira (F)':<11} | {'Área (A)':<8} | {'Estacas (S)':<11}")
print("-" * 105)

# Linhas da tabela
for i in range(1, N + 1):
    for j in range(1, M[i] + 1):
        print(f"Floresta {i:<1} | Lote {j:<1} | {P[(i,j)]:<9} | {W[(i,j)]:<12} | {T[(i,j)]:<11} | {F[(i,j)]:<11} | {A[(i,j)]:<8} | {S[(i,j)]:<11}")
print("=" * 105)
print("\n")

# ==========================================
# 3. CONSTRUÇÃO DO MODELO NO PULP
# ==========================================
model = pulp.LpProblem("Otimizacao_Florestal", pulp.LpMaximize)

# Variáveis de decisão
x = pulp.LpVariable.dicts("x", ((i, j) for i in range(1, N + 1) for j in range(1, M[i] + 1)), cat='Binary')

# Função Objetivo
model += pulp.lpSum(P[(i, j)] * x[(i, j)] for i in range(1, N + 1) for j in range(1, M[i] + 1)), "Lucro_Total"

# Restrições
for i in range(1, N + 1):
    model += pulp.lpSum(A[(i, j)] * x[(i, j)] for j in range(1, M[i] + 1)) <= A_max[i], f"Area_Max_Floresta_{i}"

model += pulp.lpSum(W[(i, j)] * x[(i, j)] for i in range(1, N + 1) for j in range(1, M[i] + 1)) <= MO_max, "Mao_de_Obra_Max"
model += pulp.lpSum(T[(i, j)] * x[(i, j)] for i in range(1, N + 1) for j in range(1, M[i] + 1)) <= NC_max, "Transporte_Max"
model += pulp.lpSum(F[(i, j)] * x[(i, j)] for i in range(1, N + 1) for j in range(1, M[i] + 1)) >= SL_min, "Suprimento_Min_Madeira"
model += pulp.lpSum(S[(i, j)] * x[(i, j)] for i in range(1, N + 1) for j in range(1, M[i] + 1)) >= EM_min, "Estacas_Mouroes_Min"

# ==========================================
# 4. RESOLUÇÃO E EXIBIÇÃO DOS RESULTADOS
# ==========================================
status = model.solve()

print(f"Status da Solução: {pulp.LpStatus[status]}")

if pulp.LpStatus[status] == "Optimal":
    print(f"\nLucro Total Maximizado: R$ {pulp.value(model.objective):.2f}\n")
    print("Lotes Escolhidos:")
    print("-" * 30)
    for i in range(1, N + 1):
        for j in range(1, M[i] + 1):
            if x[(i, j)].varValue == 1:
                print(f"-> Floresta {i}, Lote {j} foi SELECIONADO.")
else:
    print("\nNão foi possível encontrar uma solução ótima que respeite todas as restrições.")

                     TABELA DE PARÂMETROS DOS LOTES                     
Floresta   | Lote   | Lucro (P) | Mão-Obra (W) | Transp. (T) | Madeira (F) | Área (A) | Estacas (S)
---------------------------------------------------------------------------------------------------------
Floresta 1 | Lote 1 | 131       | 8            | 2           | 65          | 25       | 29         
Floresta 1 | Lote 2 | 67        | 8            | 10          | 41          | 47       | 42         
Floresta 1 | Lote 3 | 54        | 5            | 3           | 57          | 24       | 47         
Floresta 1 | Lote 4 | 127       | 5            | 10          | 55          | 44       | 41         
Floresta 2 | Lote 1 | 78        | 19           | 6           | 30          | 20       | 59         
Floresta 2 | Lote 2 | 104       | 15           | 6           | 49          | 23       | 36         
Floresta 2 | Lote 3 | 63        | 7            | 8           | 42          | 32       | 37         
Floresta 3 | Lote 1 |